Nous allons, dans ce fichier, mettre en place l'algo de PPO pour notre agent RL. 

In [1]:
!pip install pandas

  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.3.5-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl (11.3 MB)
Using cached numpy-2.3.5-cp311-cp311-win_amd64.whl (13.1 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------- ----------------------------- 1/4 [tzdata]
   ---------- ----------------------------- 1/4 [tzdata]
   ---------- ----------------------------- 1/4 [tzdata]
   ---------- ----------------------------- 

In [3]:
!pip install gymnasium

  Using cached gymnasium-1.2.2-py3-none-any.whl.metadata (10 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached Farama_Notifications-0.0.4-py3-none-any.whl.metadata (558 bytes)
Using cached gymnasium-1.2.2-py3-none-any.whl (952 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached Farama_Notifications-0.0.4-py3-none-any.whl (2.5 kB)

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   -------------------------- ------------- 2/3 [gymnasium]
   ----

In [37]:
!pip install gym_trading_env

Exception ignored in: <_io.FileIO name=3 mode='wb' closefd=True>
Traceback (most recent call last):
  File "c:\Users\mathi\OneDrive\Bureau\Cours 5a\RL\Projet\reinforcement-learning-project\.venv\Lib\site-packages\IPython\utils\_process_win32.py", line 138, in system
    res = process_handler(cmd, _system_body)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Exception ignored in: <_io.FileIO name=4 mode='rb' closefd=True>
Traceback (most recent call last):
  File "c:\Users\mathi\OneDrive\Bureau\Cours 5a\RL\Projet\reinforcement-learning-project\.venv\Lib\site-packages\IPython\utils\_process_win32.py", line 138, in system
    res = process_handler(cmd, _system_body)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Exception ignored in: <_io.FileIO name=5 mode='rb' closefd=True>
Traceback (most recent call last):
  File "c:\Users\mathi\OneDrive\Bureau\Cours 5a\RL\Projet\reinforcement-learning-project\.venv\Lib\site-packages\IPython\utils\_process_win32.py", line 138, in system
    res = process_h

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env

In [61]:
def preprocess(df):
    df = df.sort_index()
    df = df.dropna()
    df = df.drop_duplicates()

    df['feature_close'] = (df['close'] - df['close'].mean()) / df['close'].std()

    return df

df = preprocess(pd.read_pickle('./data/yfinance-AAPL-1h.pkl'))
df.head(20)

,open,high,low,close,volume,date_close,feature_close
date_open,,,,,,,
2023-11-21 14:30:00,191.410004,191.520004,190.270004,190.347504,8992766,2023-11-21 15:30:00,-0.953609
2023-11-21 15:30:00,190.350006,190.470001,189.740005,189.942001,5681421,2023-11-21 16:30:00,-0.969213
2023-11-21 16:30:00,189.940002,190.320007,189.830002,190.274994,3359800,2023-11-21 17:30:00,-0.956399
2023-11-21 17:30:00,190.276993,190.600006,190.039993,190.119995,3112369,2023-11-21 18:30:00,-0.962363
2023-11-21 18:30:00,190.119995,190.615005,190.085007,190.380005,2874142,2023-11-21 19:30:00,-0.952358
2023-11-21 19:30:00,190.382797,190.710007,190.289993,190.649994,3442863,2023-11-21 20:30:00,-0.941969
2023-11-21 20:30:00,190.645004,190.729996,190.289993,190.660004,5535871,2023-11-21 21:30:00,-0.941583
2023-11-22 14:30:00,192.360001,192.929993,191.429993,191.931503,11546645,2023-11-22 15:30:00,-0.892655
2023-11-22 15:30:00,191.949997,192.369995,191.389999,191.660004,4786188,2023-11-22 16:30:00,-0.903103


In [52]:
env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    # LIGNES SUIVANTES :
    # Valeur initiale du portefeuille
    portfolio_initial_value=1_000,
    # Frais de transactions
    trading_fees=0.1/100,
    # Intérêts sur les prêts
    borrow_interest_rate=0.02/100/24,
)

In [62]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.env_checker import check_env

#print(check_env(env=env))

#vec_env = make_vec_env(env, n_envs=4)

model = PPO("MlpPolicy",env)

model.learn(total_timesteps=250000)
model.save("PPO_test1")


Market Return : 904.05%   |   Portfolio Return : -99.95%   |   Portfolio Valuation : 0.51   |   
Market Return :  5.27%   |   Portfolio Return : -6.80%   |   Portfolio Valuation : 931.98   |   
Market Return : 44.45%   |   Portfolio Return : 46.54%   |   Portfolio Valuation : 1465.42   |   
Market Return : 103.26%   |   Portfolio Return : 107.76%   |   Portfolio Valuation : 2077.65   |   
Market Return : 10.54%   |   Portfolio Return :  7.64%   |   Portfolio Valuation : 1076.41   |   
Market Return : 27.70%   |   Portfolio Return : 27.51%   |   Portfolio Valuation : 1275.09   |   
Market Return : 677.81%   |   Portfolio Return : 667.93%   |   Portfolio Valuation : 7679.33   |   
Market Return : 39.94%   |   Portfolio Return : 39.94%   |   Portfolio Valuation : 1399.44   |   
Market Return : 103.26%   |   Portfolio Return : 103.01%   |   Portfolio Valuation : 2030.12   |   
Market Return : 904.05%   |   Portfolio Return : 873.08%   |   Portfolio Valuation : 9730.76   |   
Market Return 

In [ ]:
i=0
obs,_ = env.reset()
while i<100:
    i+=1
    action, _states = model.predict(obs)
    print(action)
    obs, reward, terminated, truncated, info = env.step(action)


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.nd

In [ ]:
nb_episodes = 100

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)


for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    print(f'Episode n˚{episode} -- Jeu de donnée {env.name}')
    done = False

    while not done:
        action, _states = model.predict(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    env.save_for_render(dir="logs")        
    if terminated:
        print('Argent perdu')
        
    elif truncated:
        print('Épisode terminé')
    

Episode n˚1 -- Jeu de donnée yfinance-GOLDUSD-1h.pkl
Market Return : 103.26%   |   Portfolio Return : 103.26%   |   Portfolio Valuation : 2032.56   |   
Épisode terminé
Episode n˚2 -- Jeu de donnée yfinance-EURUSD-1h.pkl
Market Return :  5.27%   |   Portfolio Return :  5.45%   |   Portfolio Valuation : 1054.49   |   
Épisode terminé
Episode n˚3 -- Jeu de donnée yfinance-CAC40-1h.pkl
Market Return : 10.54%   |   Portfolio Return : 10.26%   |   Portfolio Valuation : 1102.62   |   
Épisode terminé
Episode n˚4 -- Jeu de donnée binance-ETHUSD-1h.pkl
Market Return : 904.05%   |   Portfolio Return : 905.27%   |   Portfolio Valuation : 10052.66   |   
Épisode terminé
Episode n˚5 -- Jeu de donnée yfinance-AAPL-1h.pkl
Market Return : 39.94%   |   Portfolio Return : 39.94%   |   Portfolio Valuation : 1399.44   |   
Épisode terminé
Episode n˚6 -- Jeu de donnée yfinance-S&P500-1h.pkl
Market Return : 44.45%   |   Portfolio Return : 44.45%   |   Portfolio Valuation : 1444.5   |   
Épisode terminé
Epi

KeyboardInterrupt: 

In [67]:
env.historical_info[-1]

{'idx': 2763,
 'step': 2763,
 'date': np.datetime64('2025-06-25T13:30:00.000000000'),
 'position_index': array(1),
 'position': 1,
 'real_position': np.float64(1.0000000000000002),
 'data_date_close': Timestamp('2025-06-25 14:30:00'),
 'data_close': 6100.830078125,
 'data_high': 6108.509765625,
 'data_volume': 0,
 'data_open': 6104.22998046875,
 'data_low': 6096.8798828125,
 'portfolio_valuation': np.float64(1346.3183462273655),
 'portfolio_distribution_asset': np.float64(0.22067789612018449),
 'portfolio_distribution_fiat': 0,
 'portfolio_distribution_borrowed_asset': 0,
 'portfolio_distribution_borrowed_fiat': np.float64(1.1368683772161603e-13),
 'portfolio_distribution_interest_asset': 0.0,
 'portfolio_distribution_interest_fiat': np.float64(9.473903143468003e-19),
 'reward': np.float64(0.0012383509773613998)}